# WiDS 2026 Global Datathon 
Evaluation Metric:
    Hybrid Score = 0.3 × C-index + 0.7 × (1 - Weighted Brier Score)
    Weighted Brier = 0.3 × Brier@24h + 0.4 × Brier@48h + 0.3 × Brier@72h




In [1]:
# ── Run this once to install dependencies ──
!pip install lifelines scikit-survival -q

## PART 0 — IMPORTS & SETUP

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import sys

from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.isotonic import IsotonicRegression

from lifelines import CoxPHFitter, KaplanMeierFitter
from lifelines.statistics import logrank_test

from sksurv.ensemble import (
    GradientBoostingSurvivalAnalysis,
    RandomSurvivalForest,
)
from sksurv.metrics import concordance_index_censored

warnings.filterwarnings("ignore")

# Plot styling
PURPLE = "#7B2D8E"
GREEN = "#2E8B57"
plt.rcParams.update({"figure.figsize": (12, 5), "font.size": 11})

## PART 1 — LOAD DATA

In [3]:
print("=" * 80)
print("PART 1 — LOADING DATA")
print("=" * 80)

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
sample = pd.read_csv("sample_submission.csv")

print(f"Train: {train.shape}  |  Test: {test.shape}  |  Sample: {sample.shape}")
print(f"Event distribution: {train['event'].value_counts().to_dict()}")
print(f"Event rate: {train['event'].mean():.3f}")
print(f"Time range: [{train['time_to_hit_hours'].min():.1f}, {train['time_to_hit_hours'].max():.1f}] hours")

ALL_FEATURES = [
    c for c in train.columns
    if c not in ["event_id", "time_to_hit_hours", "event"]
]
print(f"Original features: {len(ALL_FEATURES)}")

PART 1 — LOADING DATA


FileNotFoundError: [Errno 2] No such file or directory: 'train.csv'

## PART 2 — FEATURE SELECTION ANALYSIS

In [ ]:
print("\n" + "=" * 80)
print("PART 2 — FEATURE SELECTION ANALYSIS")
print("=" * 80)

hit_df = train[train["event"] == 1]
miss_df = train[train["event"] == 0]
print(f"Hit fires: {len(hit_df)}  |  Censored fires: {len(miss_df)}")

In [ ]:
# ── Step 1: Separation & correlation scoring ────────────────────────────────
rows = []
for col in ALL_FEATURES:
    h = hit_df[col].mean()
    m = miss_df[col].mean()
    denom = (abs(h) + abs(m)) / 2
    sep = abs(h - m) / denom * 100 if denom != 0 else 0
    corr_event = abs(train[col].corr(train["event"]))
    corr_time = abs(train[col].corr(train["time_to_hit_hours"]))
    corr_timing_hit = abs(hit_df[col].corr(hit_df["time_to_hit_hours"]))
    rows.append({
        "feature": col,
        "sep_pct": round(sep, 1),
        "corr_event": round(corr_event, 4),
        "corr_time": round(corr_time, 4),
        "corr_timing_hit": round(corr_timing_hit, 4),
    })

scores = pd.DataFrame(rows).sort_values("corr_event", ascending=False)
print("\nFeature Scores (sorted by correlation with event):")
print(scores.to_string(index=False))

In [ ]:
# ── Step 2: Select top 10 features ──────────────────────────────────────────
TOP_10 = [
    "dist_min_ci_0_5h",        # dominant predictor — distance to evac zone
    "alignment_abs",           # fire-to-zone alignment (direction)
    "log1p_growth",            # fire area growth (log)
    "num_perimeters_0_5h",     # number of observed perimeters
    "dt_first_last_0_5h",      # time span of observations
    "spread_bearing_cos",      # cosine of spread direction
    "area_growth_rate_ha_per_h",  # area growth rate
    "radial_growth_rate_m_per_h", # radial growth rate
    "dist_fit_r2_0_5h",        # distance fit quality
    "area_first_ha",           # initial fire area
]

In [ ]:
# ── Step 3: Check collinearity ──────────────────────────────────────────────
corr_matrix = train[TOP_10].corr().abs()
print("\nHigh pairwise correlations (|r| > 0.7) among TOP_10:")
for i in range(len(TOP_10)):
    for j in range(i + 1, len(TOP_10)):
        r = corr_matrix.iloc[i, j]
        if r > 0.7:
            print(f"  {TOP_10[i]} vs {TOP_10[j]}: r = {r:.3f}")

## PART 3 — FEATURE ENGINEERING

In [ ]:
print("\n" + "=" * 80)
print("PART 3 — FEATURE ENGINEERING")
print("=" * 80)

for df in [train, test]:
    # Log transforms for skewed features
    df["log_dist_min"] = np.log1p(df["dist_min_ci_0_5h"])

    # Interaction: distance × alignment (close AND aimed at = dangerous)
    df["dist_x_align"] = df["dist_min_ci_0_5h"] * df["alignment_abs"]

    # Ratio: how fast closing relative to distance
    df["closing_ratio"] = df["closing_speed_m_per_h"] / (df["dist_min_ci_0_5h"] + 1)

    # Binary threshold features (informed by KM analysis)
    df["within_5km"] = (df["dist_min_ci_0_5h"] <= 5000).astype(int)
    df["within_10km"] = (df["dist_min_ci_0_5h"] <= 10000).astype(int)

    # Combined risk flag: close AND fire aimed at zone
    df["close_and_aimed"] = (
        (df["dist_min_ci_0_5h"] <= 10000) & (df["alignment_abs"] > 0.3)
    ).astype(int)

    # Estimated time of arrival at current closing speed
    df["eta_hours"] = (
        df["dist_min_ci_0_5h"] / df["closing_speed_m_per_h"].clip(lower=1)
    ).clip(upper=200)

    # Growth × alignment interaction
    df["growth_x_align"] = df["log1p_growth"] * df["alignment_abs"]

# Define feature sets
TOP_FE = TOP_10 + [
    "log_dist_min", "dist_x_align", "closing_ratio",
    "within_5km", "within_10km", "close_and_aimed",
    "eta_hours", "growth_x_align",
]

# Also define trimmed set for Cox PH
TRIMMED_4 = [
    "dist_min_ci_0_5h",   # p<0.005, HR=0.07
    "alignment_abs",       # p<0.005, HR=1.61
    "num_perimeters_0_5h", # p=0.12,  HR=1.29
    "log1p_growth",        # p=0.78,  HR=1.04
]

print(f"TOP_10: {len(TOP_10)} features")
print(f"TOP_FE: {len(TOP_FE)} features (with engineered)")
print(f"Engineered features: {[f for f in TOP_FE if f not in TOP_10]}")

## PART 4 — KAPLAN-MEIER CURVES

In [ ]:

print("\n--- KM Curves ---")
kmf = KaplanMeierFitter()
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()
all_km_outputs = []

# 4. Safe function
def plot_and_test(data, group_col, labels, ax, title):
    global all_km_outputs
    valid_groups = {}
    for label, subset in labels.items():
        subset = subset.dropna(subset=['time_to_hit_hours', 'event'])
        if subset.empty:
            print(f"[WARNING] Skipping {label} in {title} (empty group)")
            continue
        valid_groups[label] = subset

        kmf.fit(durations=subset['time_to_hit_hours'], 
                event_observed=subset['event'], 
                label=label)
        kmf.plot_survival_function(ax=ax)

        # Extract KM data
        km_data = kmf.survival_function_.reset_index()
        ci_data = kmf.confidence_interval_.reset_index()
        km_output = km_data.merge(ci_data, on='timeline') # Added quotes
        km_output.columns = ['time', 'survival_prob', 'ci_lower', 'ci_upper']
        km_output['group'] = label
        km_output['analysis'] = title
        all_km_outputs.append(km_output)

    if len(valid_groups) >= 2:
        g1, g2 = list(valid_groups.values())[:2]
        results = logrank_test(g1['time_to_hit_hours'], g2['time_to_hit_hours'], 
                               event_observed_A=g1['event'], event_observed_B=g2['event'])
        ax.set_title(f"{title}\nLog-Rank p-value: {results.p_value:.4f}")
    else:
        ax.set_title(f"{title}\n(Not enough data)")
    ax.set_ylim(0, 1.05)

# 5. Feature splits (Check column names match your CSV exactly)
features = {
    0: ('dist_min_ci_0_5h', 'Min Distance'),
    1: ('closing_speed_m_per_h', 'Closing Speed'),
    2: ('alignment_abs', 'Alignment'),
    3: ('area_growth_rate_ha_per_h', 'Growth Rate')
}

for i, (col, title) in features.items():
    if col in df.columns:
        med = df[col].median()
        # Custom logic for Distance (Quartiles) as per your original snippet
        if col == 'dist_min_ci_0_5h':
            q1, q4 = df[col].quantile([0.25, 0.75])
            labels = {"Closest (Q1)": df[df[col] <= q1], "Farthest (Q4)": df[df[col] >= q4]}
        else:
            labels = {f"High/Fast {title}": df[df[col] > med], f"Low/Slow {title}": df[df[col] <= med]}
        
        plot_and_test(df, col, labels, axes[i], title)
    else:
        print(f"[ERROR] Column {col} not found")

plt.tight_layout()
plt.show()

# 7. Save
if all_km_outputs:
    final_km_df = pd.concat(all_km_outputs, ignore_index=True)
    final_km_df.to_csv('km_curve_all_results.csv', index=False)
    print("KM data saved.")


## PART 5 — COX PH MODEL (BASELINE)

In [ ]:
print("\n" + "=" * 80)
print("PART 5 — COX PH MODEL (BASELINE)")
print("=" * 80)

# Fit Cox PH with trimmed 4 features (pen=0.01)
df_cox = train[TRIMMED_4 + ["time_to_hit_hours", "event"]].copy()
scaler_cox = StandardScaler()
df_cox[TRIMMED_4] = scaler_cox.fit_transform(df_cox[TRIMMED_4])

cph = CoxPHFitter(penalizer=0.01)
cph.fit(df_cox, duration_col="time_to_hit_hours", event_col="event")
print(f"Cox PH C-index: {cph.concordance_index_:.4f}")
cph.print_summary()

# Generate Cox PH submission
test_cox = test[TRIMMED_4].copy()
test_cox[TRIMMED_4] = scaler_cox.transform(test_cox[TRIMMED_4])
sf_cox = cph.predict_survival_function(test_cox)


def prob_hit_by_cox(sf_df, t):
    """Extract P(hit by time t) from Cox survival function."""
    idx = np.searchsorted(sf_df.index.values, t, side="right") - 1
    idx = max(0, min(idx, len(sf_df) - 1))
    return np.clip(1 - sf_df.iloc[idx].values, 0, 1)


submission_cox = pd.DataFrame({
    "event_id": test["event_id"].values,
    "prob_12h": prob_hit_by_cox(sf_cox, 12),
    "prob_24h": prob_hit_by_cox(sf_cox, 24),
    "prob_48h": prob_hit_by_cox(sf_cox, 48),
    "prob_72h": prob_hit_by_cox(sf_cox, 72),
})
submission_cox.to_csv("submission_cox_baseline.csv", index=False)
print("Saved: submission_cox_baseline.csv")

## PART 6 — HELPER FUNCTIONS

In [ ]:
def make_surv_y(df):
    """Convert DataFrame to scikit-survival structured array."""
    return np.array(
        [(bool(e), t) for e, t in zip(df["event"], df["time_to_hit_hours"])],
        dtype=[("event", bool), ("time", float)],
    )


def safe_eval_sf(sf_funcs, t):
    """Safely evaluate survival functions at time t, handling domain issues."""
    probs = []
    for sf in sf_funcs:
        try:
            p = 1 - sf(t)
        except ValueError:
            p = 1 - sf(min(t, sf.domain[1]))
        probs.append(np.clip(p, 0, 1))
    return np.array(probs)


PROB_COLS = ["prob_12h", "prob_24h", "prob_48h", "prob_72h"]


def ensure_monotonic(df):
    """Ensure prob_12h ≤ prob_24h ≤ prob_48h ≤ prob_72h for every row."""
    df = df.copy()
    for i in range(len(df)):
        for j in range(1, len(PROB_COLS)):
            if df.loc[i, PROB_COLS[j]] < df.loc[i, PROB_COLS[j - 1]]:
                df.loc[i, PROB_COLS[j]] = df.loc[i, PROB_COLS[j - 1]]
    return df


def make_submission(sf_funcs, event_ids):
    """Build a submission DataFrame from survival functions."""
    sub = pd.DataFrame({"event_id": event_ids})
    for t in [12, 24, 48, 72]:
        sub[f"prob_{t}h"] = safe_eval_sf(sf_funcs, t)
    return ensure_monotonic(sub)


def censor_aware_brier(y_time, y_event, pred_probs, horizon):
    """
    Censor-aware Brier score at a given horizon (competition metric).
    - Hits (event=1, time ≤ horizon): true label = 1
    - Hits (event=1, time > horizon): true label = 0
    - Censored after horizon (event=0, time ≥ horizon): true label = 0
    - Censored before horizon (event=0, time < horizon): EXCLUDED
    """
    included, labels = [], []
    for i in range(len(y_time)):
        if y_event[i] == 1:
            labels.append(1 if y_time[i] <= horizon else 0)
            included.append(i)
        elif y_time[i] >= horizon:
            labels.append(0)
            included.append(i)
    if not included:
        return 0.0
    return np.mean((pred_probs[included] - np.array(labels)) ** 2)


def weighted_brier(y_time, y_event, pred_24, pred_48, pred_72):
    """Weighted Brier Score = 0.3×B@24 + 0.4×B@48 + 0.3×B@72."""
    b24 = censor_aware_brier(y_time, y_event, pred_24, 24)
    b48 = censor_aware_brier(y_time, y_event, pred_48, 48)
    b72 = censor_aware_brier(y_time, y_event, pred_72, 72)
    return 0.3 * b24 + 0.4 * b48 + 0.3 * b72


def hybrid_score(y_time, y_event, risk_scores, pred_24, pred_48, pred_72):
    """
    Exact competition metric:
    Hybrid = 0.3 × C-index + 0.7 × (1 - Weighted Brier Score)
    """
    y_surv = make_surv_y(
        pd.DataFrame({"event": y_event, "time_to_hit_hours": y_time})
    )
    ci = concordance_index_censored(y_surv["event"], y_surv["time"], risk_scores)[0]
    wb = weighted_brier(y_time, y_event, pred_24, pred_48, pred_72)
    return 0.3 * ci + 0.7 * (1 - wb), ci, wb

## PART 7 — CROSS-VALIDATED MODEL COMPARISON ON HYBRID SCORE

In [ ]:
print("\n--- Cross Validation ---")
group_df = df.groupby(GROUP_COL)[TARGET_COL].mean().reset_index()

# Convert to binary (0 or 1)
group_df[TARGET_COL] = (group_df[TARGET_COL] > 0).astype(int)

# 4. Create folds using StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

group_df["fold"] = -1

for fold, (_, val_idx) in enumerate(skf.split(group_df, group_df[TARGET_COL])):
    group_df.loc[val_idx, "fold"] = fold

# 5. Map folds back to original data
df = df.merge(group_df[[GROUP_COL, "fold"]], on=GROUP_COL, how="left")

# 6. (Optional) Check distribution
print(df["fold"].value_counts())
print(df.groupby("fold")[TARGET_COL].mean())

In [ ]:
# ── Random Survival Forest ──────────────────────────────────────────────────
print("\n--- Random Survival Forest ---")
cv_hybrid(X_fe, RandomSurvivalForest,
          dict(n_estimators=200, min_samples_leaf=5, max_features="sqrt",
               random_state=42, n_jobs=-1),
          "RSF n=200 leaf=5 top_FE")
cv_hybrid(X_fe, RandomSurvivalForest,
          dict(n_estimators=200, min_samples_leaf=7, max_features="sqrt",
               random_state=42, n_jobs=-1),
          "RSF n=200 leaf=7 top_FE")
cv_hybrid(X_fe, RandomSurvivalForest,
          dict(n_estimators=500, min_samples_leaf=5, max_features="sqrt",
               random_state=42, n_jobs=-1),
          "RSF n=500 leaf=5 top_FE")
cv_hybrid(X_10, RandomSurvivalForest,
          dict(n_estimators=500, min_samples_leaf=5, max_features="sqrt",
               random_state=42, n_jobs=-1),
          "RSF n=500 leaf=5 top_10")

In [ ]:
# ── Gradient Boosted Survival Analysis ──────────────────────────────────────
print("\n--- Gradient Boosted Survival ---")
cv_hybrid(X_fe, GradientBoostingSurvivalAnalysis,
          dict(n_estimators=100, learning_rate=0.05, max_depth=3,
               min_samples_leaf=5, subsample=0.8, random_state=42),
          "GBSA n=100 lr=.05 d=3 leaf=5 top_FE")
cv_hybrid(X_fe, GradientBoostingSurvivalAnalysis,
          dict(n_estimators=80, learning_rate=0.05, max_depth=4,
               min_samples_leaf=7, subsample=0.7, random_state=42),
          "GBSA n=80 lr=.05 d=4 leaf=7 top_FE")
cv_hybrid(X_10, GradientBoostingSurvivalAnalysis,
          dict(n_estimators=100, learning_rate=0.05, max_depth=3,
               min_samples_leaf=5, subsample=0.8, random_state=42),
          "GBSA n=100 lr=.05 d=3 leaf=5 top_10")

# Sort and display
res_df = pd.DataFrame(results).sort_values("hybrid", ascending=False)
print(f"\n{'Model':<50} {'Hybrid':>8} {'C-idx':>7} {'W-Brier':>8}")
print("-" * 77)
for _, r in res_df.iterrows():
    print(
        f"{r['model']:<50} {r['hybrid']:>8.4f} "
        f"{r['c_index']:>7.4f} {r['w_brier']:>8.5f}"
    )
print("\nKey insight: RSF has best calibration (lowest Brier).")
print("GBSA has slightly better C-index, but 70% of the score is Brier.")

## PART 8 — ISOTONIC CALIBRATION (cross-validated, no data leakage)

In [ ]:
print("\n" + "=" * 80)
print("PART 8 — ISOTONIC CALIBRATION")
print("=" * 80)

# Step 1: Collect out-of-fold predictions from RSF
oof_preds = {t: np.zeros(len(train)) for t in [24, 48, 72]}
oof_labels = {t: np.full(len(train), np.nan) for t in [24, 48, 72]}

for tr_idx, va_idx in kf.split(X_fe):
    rsf = RandomSurvivalForest(
        n_estimators=200, min_samples_leaf=5, max_features="sqrt",
        random_state=42, n_jobs=-1,
    )
    rsf.fit(X_fe[tr_idx], y_train_struct[tr_idx])
    sf = rsf.predict_survival_function(X_fe[va_idx])

    for t in [24, 48, 72]:
        oof_preds[t][va_idx] = safe_eval_sf(sf, t)
        for i, idx in enumerate(va_idx):
            yt = train.iloc[idx]["time_to_hit_hours"]
            ye = train.iloc[idx]["event"]
            if ye == 1:
                oof_labels[t][idx] = 1 if yt <= t else 0
            elif yt >= t:
                oof_labels[t][idx] = 0
            # else: censored before horizon — leave as NaN (excluded)

# Step 2: Fit isotonic regression on OOF predictions
iso_models = {}
for t in [24, 48, 72]:
    valid = ~np.isnan(oof_labels[t])
    n_valid = valid.sum()
    n_pos = int(oof_labels[t][valid].sum())
    pct_pos = n_pos / n_valid * 100

    print(f"\n  Horizon {t}h: {n_valid} valid, {n_pos} positives ({pct_pos:.1f}%)")

    brier_raw = np.mean((oof_preds[t][valid] - oof_labels[t][valid]) ** 2)

    if pct_pos >= 95 or n_pos < 10:
        # Too extreme — isotonic will overfit, skip calibration
        print(f"    → SKIP calibration (extreme class balance)")
        iso_models[t] = None
    else:
        iso = IsotonicRegression(out_of_bounds="clip", y_min=0.001, y_max=0.999)
        iso.fit(oof_preds[t][valid], oof_labels[t][valid])
        cal = iso.predict(oof_preds[t][valid])
        brier_cal = np.mean((cal - oof_labels[t][valid]) ** 2)
        improvement = (brier_raw - brier_cal) / brier_raw * 100
        print(f"    → Brier: {brier_raw:.5f} → {brier_cal:.5f} ({improvement:.1f}% better)")
        iso_models[t] = iso

calibrated_horizons = [t for t in [24, 48, 72] if iso_models[t] is not None]
print(f"\nCalibrating horizons: {calibrated_horizons}")

## PART 9 — GBSA FEATURE IMPORTANCE

In [ ]:
print("\n" + "=" * 80)
print("PART 9 — FEATURE IMPORTANCE")
print("=" * 80)

gbs_fi = GradientBoostingSurvivalAnalysis(
    n_estimators=100, learning_rate=0.05, max_depth=3,
    min_samples_leaf=5, subsample=0.8, random_state=42,
)
gbs_fi.fit(X_fe, y_train_struct)

fi = pd.DataFrame({
    "feature": TOP_FE,
    "importance": gbs_fi.feature_importances_,
}).sort_values("importance", ascending=False)

print("\nGBSA Feature Importance:")
print(fi.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(fi["feature"][::-1], fi["importance"][::-1], color=PURPLE, alpha=0.85)
ax.set_xlabel("Feature Importance")
ax.set_title("GBSA Feature Importance (TOP_FE)")
plt.tight_layout()
plt.savefig("feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: feature_importance.png")

## PART 10 — GENERATE ALL SUBMISSIONS

In [ ]:
print("\n" + "=" * 80)
print("PART 10 — GENERATING SUBMISSIONS")
print("=" * 80)

X_tr_fe = train[TOP_FE].values
X_te_fe = test[TOP_FE].values

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Submission 1: RSF 7-seed ensemble (raw)

In [ ]:
# Submission 1: RSF 7-seed ensemble (raw)
# ──────────────────────────────────────────────────────────────────────────────
print("\n--- Building RSF 7-seed ensemble ---")
rsf_subs = []
RSF_SEEDS = [42, 123, 456, 789, 2024, 1337, 9999]

for seed in RSF_SEEDS:
    rsf = RandomSurvivalForest(
        n_estimators=200, min_samples_leaf=5, max_features="sqrt",
        random_state=seed, n_jobs=-1,
    )
    rsf.fit(X_tr_fe, y_train_struct)
    sf = rsf.predict_survival_function(X_te_fe)
    rsf_subs.append(make_submission(sf, test["event_id"].values))
    print(f"  RSF seed={seed} done")

sub_rsf_raw = pd.DataFrame({"event_id": test["event_id"].values})
for col in PROB_COLS:
    sub_rsf_raw[col] = np.mean([s[col].values for s in rsf_subs], axis=0)
sub_rsf_raw = ensure_monotonic(sub_rsf_raw)
sub_rsf_raw.to_csv("submission_rsf7_raw.csv", index=False)
print("Saved: submission_rsf7_raw.csv")

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Submission 2: RSF 7-seed + isotonic calibration (24h & 48h only)

In [ ]:
# Submission 2: RSF 7-seed + isotonic calibration (24h & 48h only)
# ──────────────────────────────────────────────────────────────────────────────
print("\n--- Applying isotonic calibration ---")
sub_rsf_cal = sub_rsf_raw.copy()
for t in calibrated_horizons:
    col = f"prob_{t}h"
    sub_rsf_cal[col] = iso_models[t].predict(sub_rsf_raw[col].values)
sub_rsf_cal = ensure_monotonic(sub_rsf_cal)
sub_rsf_cal.to_csv("submission_rsf7_calibrated.csv", index=False)
print("Saved: submission_rsf7_calibrated.csv")

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Submission 3: Mega ensemble (7 RSF + 3 GBSA, RSF-weighted)

In [ ]:
# Submission 3: Mega ensemble (7 RSF + 3 GBSA, RSF-weighted)
# ──────────────────────────────────────────────────────────────────────────────
print("\n--- Building mega ensemble (7 RSF + 3 GBSA) ---")
gbsa_subs = []
GBSA_SEEDS = [42, 123, 456]

for seed in GBSA_SEEDS:
    gbs = GradientBoostingSurvivalAnalysis(
        n_estimators=100, learning_rate=0.05, max_depth=3,
        min_samples_leaf=5, subsample=0.8, random_state=seed,
    )
    gbs.fit(X_tr_fe, y_train_struct)
    sf = gbs.predict_survival_function(X_te_fe)
    gbsa_subs.append(make_submission(sf, test["event_id"].values))
    print(f"  GBSA seed={seed} done")

# Weight RSF 1.5x vs GBSA 0.8x (RSF has better calibration)
all_subs = rsf_subs + gbsa_subs
all_weights = [1.5] * len(rsf_subs) + [0.8] * len(gbsa_subs)
total_w = sum(all_weights)

sub_mega_raw = pd.DataFrame({"event_id": test["event_id"].values})
for col in PROB_COLS:
    sub_mega_raw[col] = sum(
        w * s[col].values for w, s in zip(all_weights, all_subs)
    ) / total_w
sub_mega_raw = ensure_monotonic(sub_mega_raw)
sub_mega_raw.to_csv("submission_mega_raw.csv", index=False)
print("Saved: submission_mega_raw.csv")

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Submission 4: Mega ensemble + isotonic calibration

In [ ]:
# Submission 4: Mega ensemble + isotonic calibration
# ──────────────────────────────────────────────────────────────────────────────
sub_mega_cal = sub_mega_raw.copy()
for t in calibrated_horizons:
    col = f"prob_{t}h"
    sub_mega_cal[col] = iso_models[t].predict(sub_mega_raw[col].values)
sub_mega_cal = ensure_monotonic(sub_mega_cal)
sub_mega_cal.to_csv("submission_mega_calibrated.csv", index=False)
print("Saved: submission_mega_calibrated.csv")

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Submission 5: Mega + blended calibration (70% cal + 30% raw — hedge)

In [ ]:
# Submission 5: Mega + blended calibration (70% cal + 30% raw — hedge)
# ──────────────────────────────────────────────────────────────────────────────
sub_mega_blend = pd.DataFrame({"event_id": test["event_id"].values})
for col in PROB_COLS:
    sub_mega_blend[col] = 0.7 * sub_mega_cal[col] + 0.3 * sub_mega_raw[col]
sub_mega_blend = ensure_monotonic(sub_mega_blend)
sub_mega_blend.to_csv("submission_mega_blend.csv", index=False)
print("Saved: submission_mega_blend.csv")

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Submission 6: RSF-only 12-model super ensemble

In [ ]:
# Submission 6: RSF-only 12-model super ensemble
# ──────────────────────────────────────────────────────────────────────────────
print("\n--- Building 12-model RSF super ensemble ---")
extra_rsf_subs = list(rsf_subs)  # start with our 7

for seed in [314, 628, 1024, 2048, 4096]:
    rsf = RandomSurvivalForest(
        n_estimators=200, min_samples_leaf=5, max_features="sqrt",
        random_state=seed, n_jobs=-1,
    )
    rsf.fit(X_tr_fe, y_train_struct)
    sf = rsf.predict_survival_function(X_te_fe)
    extra_rsf_subs.append(make_submission(sf, test["event_id"].values))
    print(f"  RSF seed={seed} done")

sub_rsf12 = pd.DataFrame({"event_id": test["event_id"].values})
for col in PROB_COLS:
    sub_rsf12[col] = np.mean([s[col].values for s in extra_rsf_subs], axis=0)
sub_rsf12 = ensure_monotonic(sub_rsf12)
sub_rsf12.to_csv("submission_rsf12_raw.csv", index=False)
print("Saved: submission_rsf12_raw.csv")

## PART 10B — SUPER ENSEMBLE & SMART CALIBRATION

These are the additional submissions that scored best on Kaggle:
- **sub_super_ens_weighted** — 17 models, RSF-heavy weighted (scored 0.96502 = rank 888)
- **sub_mega10_smartcal** — 10 models + isotonic calibration on 24h & 48h only
- **sub_super_ens_smartcal** — 17 models + isotonic calibration
- Blend variants — 70% calibrated + 30% raw as hedges

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SUPER ENSEMBLE — 17 models (12 RSF + 5 GBSA), RSF weighted higher
# This produced the best Kaggle score so far: 0.96502
# ═══════════════════════════════════════════════════════════════════════════════

print('=== Building Super Ensemble (17 models) ===')

super_subs = []
super_weights = []

# ── 7 RSF on TOP_FE (weight 1.5 each) ──
for seed in [42, 123, 456, 789, 2024, 1337, 9999]:
    rsf = RandomSurvivalForest(
        n_estimators=200, min_samples_leaf=5, max_features='sqrt',
        random_state=seed, n_jobs=-1)
    rsf.fit(X_tr_fe, y_train_struct)
    sf = rsf.predict_survival_function(X_te_fe)
    super_subs.append(make_submission(sf, test['event_id'].values))
    super_weights.append(1.5)
    print(f'  RSF top_FE seed={seed} done')

# ── 3 RSF on TOP_FE with leaf=7 (different calibration, weight 1.3) ──
for seed in [42, 123, 456]:
    rsf = RandomSurvivalForest(
        n_estimators=200, min_samples_leaf=7, max_features='sqrt',
        random_state=seed, n_jobs=-1)
    rsf.fit(X_tr_fe, y_train_struct)
    sf = rsf.predict_survival_function(X_te_fe)
    super_subs.append(make_submission(sf, test['event_id'].values))
    super_weights.append(1.3)
    print(f'  RSF top_FE leaf=7 seed={seed} done')

# ── 2 RSF on TOP_FE_V2 (expanded engineered features, weight 1.4) ──
for df_tmp in [train, test]:
    df_tmp['log_eta'] = np.log1p(df_tmp['eta_hours'])
    df_tmp['dist_inv'] = 1.0 / (df_tmp['dist_min_ci_0_5h'] + 100)
    df_tmp['dist_x_growth'] = df_tmp['dist_min_ci_0_5h'] * df_tmp['log1p_growth']
    df_tmp['align_x_closing'] = df_tmp['alignment_abs'] * df_tmp['closing_speed_m_per_h']
    df_tmp['dist_x_perimeters'] = df_tmp['dist_min_ci_0_5h'] * df_tmp['num_perimeters_0_5h']
    df_tmp['growth_x_perimeters'] = df_tmp['log1p_growth'] * df_tmp['num_perimeters_0_5h']
    df_tmp['within_15km'] = (df_tmp['dist_min_ci_0_5h'] <= 15000).astype(int)
    df_tmp['within_20km'] = (df_tmp['dist_min_ci_0_5h'] <= 20000).astype(int)
    df_tmp['dist_bin'] = pd.cut(df_tmp['dist_min_ci_0_5h'], bins=[0,5000,10000,20000,50000,float('inf')], labels=False)
    df_tmp['closing_positive'] = (df_tmp['closing_speed_m_per_h'] > 0).astype(int)
    df_tmp['align_x_dist_inv'] = df_tmp['alignment_abs'] * df_tmp['dist_inv']
    df_tmp['threat_score'] = df_tmp['alignment_abs'] * df_tmp['closing_ratio']

TOP_FE_V2 = TOP_FE + ['log_eta', 'dist_inv', 'dist_x_growth', 'align_x_closing',
                       'dist_x_perimeters', 'growth_x_perimeters', 'within_15km',
                       'within_20km', 'dist_bin', 'closing_positive',
                       'align_x_dist_inv', 'threat_score']
X_tr_v2 = train[TOP_FE_V2].values
X_te_v2 = test[TOP_FE_V2].values

for seed in [42, 123]:
    rsf = RandomSurvivalForest(
        n_estimators=200, min_samples_leaf=5, max_features='sqrt',
        random_state=seed, n_jobs=-1)
    rsf.fit(X_tr_v2, y_train_struct)
    sf = rsf.predict_survival_function(X_te_v2)
    super_subs.append(make_submission(sf, test['event_id'].values))
    super_weights.append(1.4)
    print(f'  RSF top_FE_V2 seed={seed} done')

# ── 3 GBSA d3 on TOP_FE (weight 0.8 — worse calibration) ──
for seed in [42, 123, 456]:
    gbs = GradientBoostingSurvivalAnalysis(
        n_estimators=100, learning_rate=0.05, max_depth=3,
        min_samples_leaf=5, subsample=0.8, random_state=seed)
    gbs.fit(X_tr_fe, y_train_struct)
    sf = gbs.predict_survival_function(X_te_fe)
    super_subs.append(make_submission(sf, test['event_id'].values))
    super_weights.append(0.8)
    print(f'  GBSA d3 top_FE seed={seed} done')

# ── 2 GBSA d2 on TOP_FE (weight 0.7) ──
for seed in [42, 123]:
    gbs = GradientBoostingSurvivalAnalysis(
        n_estimators=150, learning_rate=0.03, max_depth=2,
        min_samples_leaf=5, subsample=0.8, random_state=seed)
    gbs.fit(X_tr_fe, y_train_struct)
    sf = gbs.predict_survival_function(X_te_fe)
    super_subs.append(make_submission(sf, test['event_id'].values))
    super_weights.append(0.7)
    print(f'  GBSA d2 top_FE seed={seed} done')

print(f'\nTotal models in super ensemble: {len(super_subs)}')

# ── Weighted average ──
total_w = sum(super_weights)
sub_super_ens_weighted = pd.DataFrame({'event_id': test['event_id'].values})
for col in PROB_COLS:
    sub_super_ens_weighted[col] = sum(
        w * s[col].values for w, s in zip(super_weights, super_subs)
    ) / total_w
sub_super_ens_weighted = ensure_monotonic(sub_super_ens_weighted)
sub_super_ens_weighted.to_csv('sub_super_ens_weighted.csv', index=False)
print('\nSaved: sub_super_ens_weighted.csv  ★ BEST on Kaggle (0.96502)')
print(sub_super_ens_weighted[PROB_COLS].describe().round(3))

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SMART CALIBRATION — isotonic only on 24h & 48h (skip 72h due to data issues)
# ═══════════════════════════════════════════════════════════════════════════════

print('=== Applying Smart Calibration ===')
print('Calibrating 24h and 48h only (70% of Brier weight).')
print('Skipping 72h — extreme class imbalance causes isotonic overfit.\n')

# ── Calibrate mega ensemble (from Part 10) ──
sub_mega_smartcal = sub_mega_raw.copy()
for t in calibrated_horizons:
    col = f'prob_{t}h'
    if iso_models[t] is not None:
        sub_mega_smartcal[col] = iso_models[t].predict(sub_mega_raw[col].values)
        print(f'  Calibrated mega {col}')
sub_mega_smartcal = ensure_monotonic(sub_mega_smartcal)
sub_mega_smartcal.to_csv('sub_mega10_smartcal.csv', index=False)
print('Saved: sub_mega10_smartcal.csv')

# ── Calibrate RSF-only ensemble (from Part 10) ──
sub_rsf_smartcal = sub_rsf_raw.copy()
for t in calibrated_horizons:
    col = f'prob_{t}h'
    if iso_models[t] is not None:
        sub_rsf_smartcal[col] = iso_models[t].predict(sub_rsf_raw[col].values)
sub_rsf_smartcal = ensure_monotonic(sub_rsf_smartcal)
sub_rsf_smartcal.to_csv('sub_rsf7_smartcal.csv', index=False)
print('Saved: sub_rsf7_smartcal.csv')

# ── Calibrate super ensemble ──
sub_super_smartcal = sub_super_ens_weighted.copy()
for t in calibrated_horizons:
    col = f'prob_{t}h'
    if iso_models[t] is not None:
        sub_super_smartcal[col] = iso_models[t].predict(sub_super_ens_weighted[col].values)
sub_super_smartcal = ensure_monotonic(sub_super_smartcal)
sub_super_smartcal.to_csv('sub_super_ens_smartcal.csv', index=False)
print('Saved: sub_super_ens_smartcal.csv')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# BLENDED SUBMISSIONS — 70% calibrated + 30% raw (hedge against overfit)
# ═══════════════════════════════════════════════════════════════════════════════

print('=== Building Blended Submissions ===')

# Mega blend
sub_mega_smartblend = pd.DataFrame({'event_id': test['event_id'].values})
for col in PROB_COLS:
    sub_mega_smartblend[col] = 0.7 * sub_mega_smartcal[col] + 0.3 * sub_mega_raw[col]
sub_mega_smartblend = ensure_monotonic(sub_mega_smartblend)
sub_mega_smartblend.to_csv('sub_mega10_smartblend.csv', index=False)
print('Saved: sub_mega10_smartblend.csv')

# Super blend
sub_super_smartblend = pd.DataFrame({'event_id': test['event_id'].values})
for col in PROB_COLS:
    sub_super_smartblend[col] = 0.7 * sub_super_smartcal[col] + 0.3 * sub_super_ens_weighted[col]
sub_super_smartblend = ensure_monotonic(sub_super_smartblend)
sub_super_smartblend.to_csv('sub_super_ens_smartblend.csv', index=False)
print('Saved: sub_super_ens_smartblend.csv')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# RSF-ONLY 12-MODEL ENSEMBLE (no GBSA at all)
# ═══════════════════════════════════════════════════════════════════════════════

print('=== Building 12-model RSF-only ensemble ===')

rsf12_subs = list(rsf_subs)  # 7 RSF from Part 10

for seed in [314, 628, 1024, 2048, 4096]:
    rsf = RandomSurvivalForest(
        n_estimators=200, min_samples_leaf=5, max_features='sqrt',
        random_state=seed, n_jobs=-1)
    rsf.fit(X_tr_fe, y_train_struct)
    sf = rsf.predict_survival_function(X_te_fe)
    rsf12_subs.append(make_submission(sf, test['event_id'].values))
    print(f'  RSF seed={seed} done')

sub_rsf_only_12 = pd.DataFrame({'event_id': test['event_id'].values})
for col in PROB_COLS:
    sub_rsf_only_12[col] = np.mean([s[col].values for s in rsf12_subs], axis=0)
sub_rsf_only_12 = ensure_monotonic(sub_rsf_only_12)
sub_rsf_only_12.to_csv('sub_rsf_only_12.csv', index=False)
print('Saved: sub_rsf_only_12.csv')

## PART 11 — VERIFICATION & SUMMARY

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# FINAL VERIFICATION — ALL SUBMISSION FILES
# ═══════════════════════════════════════════════════════════════════════════════

print('\n' + '=' * 90)
print('FINAL VERIFICATION — ALL SUBMISSION FILES')
print('=' * 90)

all_files = [
    'sub_super_ens_weighted.csv',
    'sub_super_ens_smartcal.csv',
    'sub_super_ens_smartblend.csv',
    'sub_mega10_smartcal.csv',
    'sub_mega10_smartblend.csv',
    'sub_rsf7_smartcal.csv',
    'sub_rsf_only_12.csv',
    'submission_mega_calibrated.csv',
    'submission_mega_raw.csv',
    'submission_mega_blend.csv',
    'submission_rsf7_raw.csv',
    'submission_rsf7_calibrated.csv',
    'submission_rsf12_raw.csv',
    'submission_cox_baseline.csv',
]

print(f'{"File":<40} {"IDs":>5} {"Mono":>5} {"Range":>6}')
print('-' * 60)
for fname in all_files:
    try:
        df = pd.read_csv(fname)
        ids_ok = set(df['event_id']) == set(sample['event_id'])
        mono = all((df[PROB_COLS[j]] >= df[PROB_COLS[j-1]]).all() for j in range(1, len(PROB_COLS)))
        rng = (df[PROB_COLS] >= 0).all().all() and (df[PROB_COLS] <= 1).all().all()
        print(f'  {fname:<38} {str(ids_ok):>5} {str(mono):>5} {str(rng):>6}')
    except FileNotFoundError:
        print(f'  {fname:<38}  NOT FOUND')

print(f'''

{'=' * 70}
KAGGLE SUBMISSION PRIORITY:
{'=' * 70}

  1. sub_super_ens_weighted.csv      ★ BEST (0.96502, rank 888)
  2. sub_super_ens_smartcal.csv       calibrated version of #1
  3. sub_super_ens_smartblend.csv     70% cal + 30% raw hedge
  4. sub_mega10_smartcal.csv          10-model calibrated
  5. sub_rsf_only_12.csv              12 RSF, no GBSA
  6. sub_rsf7_smartcal.csv            simplest calibrated
''')